In [0]:
%run "../Includes/configurations"

In [0]:
%fs ls "abfss://presentation@formula1dlneeraj.dfs.core.windows.net/"

In [0]:
drivers_df = spark.read.parquet(f"{processed_folder_path}/drivers").withColumnRenamed('name','driver_name').withColumnRenamed('nationality','driver_nationality').withColumnRenamed('number','driver_number')

In [0]:
constructors_df = spark.read.parquet(f"{processed_folder_path}/constructors").withColumnRenamed('name','team')

In [0]:
cisrcuits_df = spark.read.parquet(f"{processed_folder_path}/circuits").withColumnRenamed('location','Circuit_location').withColumnRenamed('circuit_id','circuitId')

In [0]:
races_df = spark.read.parquet(f"{processed_folder_path}/races").withColumnRenamed('name','races_name').withColumnRenamed('race_timestamp','races_date')

In [0]:
results_df = spark.read.parquet(f"{processed_folder_path}/results").withColumnRenamed('time','races_time').withColumnRenamed("driver_id", "driver_Id")\
    .withColumnRenamed("constructor_id", "constructor_Id")

In [0]:
race_circuit_df= races_df.join(cisrcuits_df, "circuitId", "inner")\
    .select(races_df.raceId,races_df.year,races_df.races_name,races_df.races_date,cisrcuits_df.Circuit_location)

In [0]:
race_result = results_df.join(race_circuit_df, results_df.race_id==race_circuit_df.raceId)\
.join(drivers_df, results_df.driver_Id==drivers_df.driver_Id)\
.join(constructors_df, results_df.constructor_Id==constructors_df.constructor_Id)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
final_df=race_result.select("year","races_name", "races_date", "Circuit_location", "driver_name", "driver_number", "driver_nationality","team", "grid", "fastest_lap", "races_time", "points").withColumn("created_date", current_timestamp())

In [0]:
final_df.filter("year = 2020 and races_name = 'Abu Dhabi Grand Prix'").orderBy(final_df.points.desc()).display()

year,races_name,races_date,Circuit_location,driver_name,driver_number,driver_nationality,team,grid,fastest_lap,races_time,points,created_date
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Valtteri Bottas,77,Finnish,Mercedes,2,40,+15.976,18.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Lewis Hamilton,44,British,Mercedes,3,37,+18.415,15.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Alexander Albon,23,Thai,Red Bull,5,42,+19.987,12.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Lando Norris,4,British,McLaren,4,53,+1:00.729,10.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Carlos Sainz,55,Spanish,McLaren,6,48,+1:05.662,8.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Daniel Ricciardo,3,Australian,Renault,11,55,+1:13.748,7.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Pierre Gasly,10,French,AlphaTauri,9,53,+1:29.718,4.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Esteban Ocon,31,French,Renault,10,47,+1:41.069,2.0,2026-07-12T11:08:54.974441Z
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00Z,Abu Dhabi,Lance Stroll,18,Canadian,Racing Point,8,41,+1:42.738,1.0,2026-07-12T11:08:54.974441Z


In [0]:
final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/race_results")


In [0]:
%fs ls "abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/"

path,name,size,modificationTime
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/_SUCCESS,_SUCCESS,0,1783854910000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/_committed_4277353578790887482,_committed_4277353578790887482,1724,1783854910000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/_started_4277353578790887482,_started_4277353578790887482,0,1783854905000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00000-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-277-1-c000.snappy.parquet,part-00000-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-277-1-c000.snappy.parquet,22583,1783854906000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00001-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-278-1-c000.snappy.parquet,part-00001-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-278-1-c000.snappy.parquet,23232,1783854906000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00002-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-279-1-c000.snappy.parquet,part-00002-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-279-1-c000.snappy.parquet,23348,1783854906000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00003-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-280-1-c000.snappy.parquet,part-00003-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-280-1-c000.snappy.parquet,23518,1783854906000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00004-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-281-1-c000.snappy.parquet,part-00004-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-281-1-c000.snappy.parquet,22964,1783854907000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00005-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-282-1-c000.snappy.parquet,part-00005-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-282-1-c000.snappy.parquet,24160,1783854907000
abfss://presentation@formula1dlneeraj.dfs.core.windows.net/race_results/part-00006-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-283-1-c000.snappy.parquet,part-00006-tid-4277353578790887482-8d0cc6ad-b001-4c84-a12b-27ada98a99f4-283-1-c000.snappy.parquet,23660,1783854907000
